# Malawi Health Facility Data Intelligence & Performance Monitoring
## Reproducible multi-year synthetic data platform

**Purpose:** demonstrate an end-to-end health reporting data engineering and analytics workflow: synthetic source generation, dimensional modelling, staging and quarantine, operational quality rules, detector evaluation, robust peer anomaly detection, longitudinal monitoring and portfolio reporting.

> **Synthetic-data notice:** all facilities, IDs, catchment populations, targets and reported values in the expanded cohort are generated. District labels are Malawi-oriented geographic labels, not actual district performance extracts. No patient-level data or real Ministry of Health / DHIS2 records are used. This notebook is a portfolio prototype, not a production health information system.

The default run creates **120 synthetic facilities × 60 months × 8 programmes × 5 indicators = 288,000 facility-month-indicator records**.

## 1. Platform architecture and analytical grain

| Layer | Main artefact | Purpose |
|---|---|---|
| Source | Reporting fact + facility dimension | Reproducible synthetic source data |
| Staging | Staged fact + quarantine | Schema enforcement, facility enrichment and structural rejection |
| Operational analytics | Monthly, facility and programme profiles | Quality issue rates without ground-truth leakage |
| Evaluation | Confusion metrics | Compare operational detectors with controlled synthetic labels |
| Statistical monitoring | MAD/IQR anomalies + monthly limits | Identify unusual peer values and retrospective issue-rate signals |
| Presentation | Streamlit + Power BI | Decision support and portfolio communication |

The reporting fact grain is one row per **Facility_ID × Month × Programme × Indicator**. `Facility_ID` is the dimension relationship key; `Record_ID` is the unique fact-row identifier. Synthetic ground-truth columns are reserved for evaluation only.

## 2. Environment and reproducibility

Run this notebook from the repository root after installing `requirements.txt`. The generator and analytics modules are the same modules used by the command-line pipeline and Streamlit workspace.

In [ ]:
from pathlib import Path
import json
import sys
import time

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from scale_data_generator import ScaledHealthDataGenerator
from supply_chain_planning import build_supply_chain_profiles
from cold_chain_planning import build_cold_chain_profiles
from equipment_maintenance import build_equipment_maintenance_profiles
from etl_pipeline import stage_reporting_data
from quality_analytics import (
    add_operational_predictions,
    evaluate_detectors,
    build_operational_profiles,
    build_district_profiles,
)
from statistical_monitoring import (
    detect_peer_anomalies,
    build_monthly_control_chart,
)

SEED = 42
FACILITY_COUNT = 120
START_MONTH = "2021-01-01"
PERIODS = 60
OUTPUT_DIR = ROOT / "data" / "warehouse"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"Expected fact records: {FACILITY_COUNT * PERIODS * 40:,}")

## 3. Generate the expanded synthetic cohort

The fixed seed makes the generated source tables reproducible. Defect categories can overlap, except that a missing reported value is not simultaneously labelled as an observable reported-value outlier. Other defect counts are therefore not additive.

In [ ]:
generator = ScaledHealthDataGenerator(
    seed=SEED,
    facility_count=FACILITY_COUNT,
    start_month=START_MONTH,
    periods=PERIODS,
)

generation_started = time.perf_counter()
fact, facilities, manifest = generator.generate()
generation_seconds = time.perf_counter() - generation_started

assert len(fact) == generator.expected_record_count
assert len(facilities) == FACILITY_COUNT
assert fact["Record_ID"].is_unique
assert not fact.duplicated(
    ["Facility_ID", "Month", "Programme", "Indicator"]
).any()

print(f"Generated {len(fact):,} fact records in {generation_seconds:.1f} seconds")
display(pd.DataFrame([manifest]).T.rename(columns={0: "Value"}))
display(facilities.head())

## 4. Source-level validation and controlled defects

Validate expected row count, natural-key uniqueness, dimensional coverage and the actual injected defect counts before staging. Ground-truth labels are not quality rules; they document where controlled defects were injected.

In [ ]:
source_profile = {
    "fact_rows": len(fact),
    "facility_rows": len(facilities),
    "districts": facilities["District"].nunique(),
    "programmes": fact["Programme"].nunique(),
    "indicators": fact["Indicator"].nunique(),
    "reporting_months": fact["Month"].nunique(),
    "unique_facility_ids_in_fact": fact["Facility_ID"].nunique(),
    "duplicate_record_ids": int(fact["Record_ID"].duplicated().sum()),
    "duplicate_natural_keys": int(fact.duplicated(
        ["Facility_ID", "Month", "Programme", "Indicator"]
    ).sum()),
    "synthetic_facilities": bool(facilities["Is_Synthetic"].all()),
}
display(pd.Series(source_profile, name="Source validation"))

injection_summary = pd.DataFrame({
    "Defect": list(manifest["injection_counts"].keys()),
    "Injected records": list(manifest["injection_counts"].values()),
})
injection_summary["Share of fact"] = injection_summary["Injected records"] / len(fact)
display(injection_summary.style.format({"Share of fact": "{:.2%}"}))

## 5. Stage records and quarantine structural failures

The staging layer normalises dates and numeric fields, enriches facility attributes from the facility dimension, and quarantines identity, relationship, period and duplicate-grain failures. Analytical defects remain in the staged table with explicit flags so imperfect reports are still available for review.

In [ ]:
staging_started = time.perf_counter()
staged, quarantine, staging_report = stage_reporting_data(fact, facilities)
staging_seconds = time.perf_counter() - staging_started

assert len(staged) + len(quarantine) == len(fact)
assert staged["Record_ID"].is_unique
assert not staged.duplicated(
    ["Facility_ID", "Month", "Programme", "Indicator"]
).any()

print(f"Staged {len(staged):,} rows and quarantined {len(quarantine):,} rows in {staging_seconds:.1f} seconds")
display(pd.json_normalize(staging_report).T.rename(columns={0: "Value"}))

## 6. Operational detector performance

Operational predictions are built from the staged reporting fields and deterministic quality flags. Synthetic ground-truth columns are read only by the evaluation function. Precision, recall and F1 here measure recovery of controlled synthetic defects; they are **not** estimates of real-world detector accuracy.

In [ ]:
detector_metrics = evaluate_detectors(staged)
display(detector_metrics.style.format({
    "precision": "{:.1%}",
    "recall": "{:.1%}",
    "f1": "{:.1%}",
    "specificity": "{:.1%}",
    "false_positive_rate": "{:.1%}",
    "accuracy": "{:.1%}",
}))

operational = add_operational_predictions(staged)
truth_columns = [column for column in staged.columns if column.startswith("Ground_Truth_")]
blind_input = staged.drop(columns=truth_columns)
blind_predictions = add_operational_predictions(blind_input)
for prediction_column in [
    "Operational_Detected_Outlier", "Operational_Outlier_Ratio",
    "Operational_Any_Detected_Issue",
]:
    pd.testing.assert_series_equal(
        operational[prediction_column], blind_predictions[prediction_column],
        check_names=False,
    )
print("Ground-truth isolation check passed: operational predictions are unchanged when all Ground_Truth_* columns are removed.")

## 7. Longitudinal, facility and programme intelligence

Profiles are aggregated from operational flags only. Facility and district rates are volume-normalised so that larger reporting volumes are not mistaken for higher risk. District-month profiles support longitudinal comparisons across the synthetic district labels.

In [ ]:
monthly_profile, facility_profile, programme_profile = build_operational_profiles(staged)
district_profile, district_monthly_profile = build_district_profiles(staged)

display(district_profile)
display(district_monthly_profile.head())
display(monthly_profile.head())
display(facility_profile.head(15))
display(programme_profile.head(20))

In [ ]:
fig = px.line(
    monthly_profile,
    x="Month",
    y="detected_issue_rate_pct",
    markers=True,
    title="Monthly detected quality issue rate",
    labels={"detected_issue_rate_pct": "Records with detected issues (%)"},
)
fig.show()

top_facilities = facility_profile.head(15).sort_values("detected_issue_rate_pct")
fig = px.bar(
    top_facilities,
    x="detected_issue_rate_pct",
    y="Facility",
    orientation="h",
    title="Facility follow-up priorities (volume-normalised)",
    labels={"detected_issue_rate_pct": "Detected issue rate (%)"},
)
fig.show()

## 8. Statistical peer anomalies and monthly monitoring

The peer detector compares values within district × programme × indicator cohorts. It uses a median absolute deviation (MAD) robust z-score, with an IQR / scale-aware fallback for degenerate distributions. Groups below the minimum sample threshold are not flagged.

Monthly three-sigma limits are calculated retrospectively from the full observed period. They are descriptive review signals, not prospective alert thresholds or a validated statistical process-control service.

In [ ]:
statistical_started = time.perf_counter()
peer_anomalies = detect_peer_anomalies(staged)
monthly_control = build_monthly_control_chart(staged)
statistical_seconds = time.perf_counter() - statistical_started

flagged_peer_anomalies = peer_anomalies.loc[peer_anomalies["Statistical_Anomaly"]].copy()
print(f"Flagged {len(flagged_peer_anomalies):,} peer anomalies in {statistical_seconds:.1f} seconds")
display(flagged_peer_anomalies.head(25))
display(monthly_control.tail(12))

In [ ]:
fig = px.line(
    monthly_control,
    x="Month",
    y=["Issue_Rate", "Centre_Line", "Upper_Control_Limit", "Lower_Control_Limit"],
    markers=True,
    title="Retrospective monthly issue-rate monitoring",
    labels={"value": "Issue rate", "Month": "Reporting month", "variable": "Series"},
)
fig.show()

## 9. Data governance and interpretation

- All source records and facility metadata in this expanded cohort are synthetic.
- District labels are geographic context only; generated facility names and metrics do not represent real facilities.
- No patient-level or personally identifiable information is generated.
- Synthetic targets are scenario parameters, not official service-delivery targets.
- Ground-truth labels are reserved for detector evaluation and excluded from operational profiles and statistical anomaly detection.
- A detected issue is a prompt to verify source documents, not proof of error or misconduct.
- Facility rates and statistical thresholds require domain validation before any operational use.
- The prototype has no role-based access control, audit trail or durable multi-user corrective-action database.

## 10. Target attainment by comparable indicator

Calculate pooled reported-to-target attainment at facility, district, programme-indicator and month-indicator grain. The denominator remains indicator-specific; unrelated indicators are never collapsed into one facility score. This is not population coverage or unique people served.


In [ ]:
from target_attainment import build_target_attainment_profiles
attainment_profiles = build_target_attainment_profiles(staged)

for profile_name, profile in attainment_profiles.items():
    print(f"{profile_name}: {len(profile):,} rows")

facility_attainment = attainment_profiles["facility_indicator"]
display(facility_attainment.sort_values("attainment_pct").head(15))

## 10. Export reproducible platform artefacts

Large generated CSVs are written under `data/warehouse/`, which is excluded from Git. The source code, tests, documentation and notebook remain version controlled. This export cell is optional if you only want to explore the in-memory analysis.

In [ ]:
raw_dir = OUTPUT_DIR / "raw"
staged_dir = OUTPUT_DIR / "staged"
analytics_dir = OUTPUT_DIR / "analytics"
for folder in (raw_dir, staged_dir, analytics_dir):
    folder.mkdir(parents=True, exist_ok=True)

fact.to_csv(raw_dir / "facility_reporting_fact.csv", index=False)
facilities.to_csv(raw_dir / "facility_dimension.csv", index=False)
(raw_dir / "generation_manifest.json").write_text(json.dumps(manifest, indent=2))

staged.to_csv(staged_dir / "facility_reporting_staged.csv", index=False)
quarantine.to_csv(staged_dir / "facility_reporting_quarantine.csv", index=False)
(staged_dir / "staging_quality_report.json").write_text(json.dumps(staging_report, indent=2))

detector_metrics.to_csv(analytics_dir / "detector_evaluation_metrics.csv", index=False)
monthly_profile.to_csv(analytics_dir / "monthly_quality_trends.csv", index=False)
facility_profile.to_csv(analytics_dir / "facility_quality_benchmark.csv", index=False)
programme_profile.to_csv(analytics_dir / "programme_indicator_quality_profile.csv", index=False)
district_profile.to_csv(analytics_dir / "district_quality_profile.csv", index=False)
district_monthly_profile.to_csv(analytics_dir / "district_monthly_quality_trends.csv", index=False)
flagged_peer_anomalies.to_csv(analytics_dir / "statistical_peer_anomalies.csv", index=False)
monthly_control.to_csv(analytics_dir / "monthly_issue_control_chart.csv", index=False)
attainment_profiles["facility_indicator"].to_csv(analytics_dir / "facility_indicator_target_attainment.csv", index=False)
attainment_profiles["district_indicator"].to_csv(analytics_dir / "district_indicator_target_attainment.csv", index=False)
attainment_profiles["programme_indicator"].to_csv(analytics_dir / "programme_indicator_target_attainment.csv", index=False)
attainment_profiles["monthly_indicator"].to_csv(analytics_dir / "monthly_indicator_target_attainment.csv", index=False)

summary = {
    "synthetic": True,
    "generation": manifest,
    "staging": staging_report,
    "detector_metrics": detector_metrics.to_dict(orient="records"),
    "peer_anomaly_records": int(len(flagged_peer_anomalies)),
    "monthly_control_review_signals": int(monthly_control["Outside_Descriptive_Limits"].sum()),
    "runtime_seconds": {
        "generation": round(generation_seconds, 2),
        "staging": round(staging_seconds, 2),
        "statistical_monitoring": round(statistical_seconds, 2),
    },
}
(analytics_dir / "expanded_platform_summary.json").write_text(json.dumps(summary, indent=2))
print(f"Saved reproducible outputs under {OUTPUT_DIR}")

## 11. Executive interpretation

Use the outputs to answer five operational questions:

1. **Can the data be trusted structurally?** Review staging yield, duplicate keys and quarantine reasons.
2. **Which quality dimensions are failing?** Compare detector precision/recall against injected synthetic labels and inspect issue volumes.
3. **Where should follow-up start?** Use volume-normalised facility rates and programme/indicator profiles.
4. **Are values unusual relative to peers?** Review MAD/IQR anomaly flags and validate them against source documents.
5. **Are quality signals changing over time?** Inspect monthly issue rates and retrospective limits.
6. **Which comparable indicators are below target?** Review facility, district and monthly attainment with valid-record denominators.

This notebook demonstrates an end-to-end engineering and analytics workflow. It does not establish clinical performance, service coverage, official district rankings or production readiness.

## Resource capacity planning and allocation scenarios\n\nThis section creates illustrative facility capacity gaps and compares three constrained allocation heuristics. All resource units and available capacity are synthetic abstractions. They are not staffing headcounts, budgets, official Malawi staffing norms or operational recommendations. Catchment population is only a planning proxy, not patient volume or service coverage. See `docs/resource_planning_methodology.md`.\n

In [ ]:
from resource_planning import build_resource_planning_profiles

resource_profiles = build_resource_planning_profiles(
    facilities, resource_pool=120, seed=42, units_per_10000_population=2.5
)
display(resource_profiles['scenario_summary'])
display(resource_profiles['facility_capacity'].head())
display(resource_profiles['scenario_allocations'].head())


In [ ]:
analytics_dir = Path('data/warehouse/analytics')\nanalytics_dir.mkdir(parents=True, exist_ok=True)\nresource_profiles['facility_capacity'].to_csv(analytics_dir / 'facility_resource_capacity.csv', index=False)\nresource_profiles['scenario_allocations'].to_csv(analytics_dir / 'resource_allocation_scenarios.csv', index=False)\nresource_profiles['scenario_summary'].to_csv(analytics_dir / 'resource_allocation_scenario_summary.csv', index=False)\n

## Commodity availability, stockout and replenishment intelligence\n\nThe simulation builds synthetic monthly stock cards for eight illustrative commodities, estimates stockout days and unmet demand, and creates a lead-time-plus-buffer replenishment worklist under an abstract budget-credit ceiling. All demand rates, receipts and unit costs are synthetic assumptions. Commodity names are examples, not an approved national list; costs are not MWK. This is not an operational inventory or procurement recommendation. See `docs/supply_chain_methodology.md`.\n

In [ ]:
supply_profiles = build_supply_chain_profiles(\n    facilities, periods=PERIODS, start_month=START_MONTH, seed=SEED,\n    budget_credits=50000, lead_time_days=30, buffer_days=15\n)\ndisplay(supply_profiles['scenario_summary'])\ndisplay(supply_profiles['replenishment_plan'].sort_values('Recent_Stockout_Days', ascending=False).head(20))\ndisplay(supply_profiles['inventory_ledger'].head())\n

In [ ]:
analytics_dir = OUTPUT_DIR / 'analytics'\nanalytics_dir.mkdir(parents=True, exist_ok=True)\nsupply_profiles['commodity_catalog'].to_csv(analytics_dir / 'commodity_catalog.csv', index=False)\nsupply_profiles['inventory_ledger'].to_csv(analytics_dir / 'facility_commodity_stock_ledger.csv', index=False)\nsupply_profiles['replenishment_plan'].to_csv(analytics_dir / 'commodity_replenishment_plan.csv', index=False)\nsupply_profiles['scenario_summary'].to_csv(analytics_dir / 'commodity_replenishment_summary.csv', index=False)\n

## Cold-chain monitoring, batch traceability and expiry intelligence

This synthetic scenario creates facility-month temperature summaries and lot-level stock records for four illustrative temperature-sensitive commodities. Batch balances follow a monthly FEFO (First Expiry, First Out) consumption convention; expiry and near-expiry quantities are retained at batch grain. The generic 2–8°C band, shelf lives and 30/60/90-day review windows are assumptions for this portfolio demonstration—not product-specific storage instructions or official Malawi standards. Monthly summaries are not a substitute for calibrated continuous loggers or product-stability decisions. See `docs/cold_chain_methodology.md`.


In [ ]:
cold_chain_profiles = build_cold_chain_profiles(
    facilities, periods=PERIODS, start_month=START_MONTH, seed=SEED
)
display(cold_chain_profiles['temperature_monitoring'].head())
display(cold_chain_profiles['expiry_risk_summary'].tail(20))
display(cold_chain_profiles['batch_ledger'].head())


In [ ]:
analytics_dir = OUTPUT_DIR / 'analytics'
analytics_dir.mkdir(parents=True, exist_ok=True)
cold_chain_profiles['cold_chain_catalog'].to_csv(analytics_dir / 'cold_chain_catalog.csv', index=False)
cold_chain_profiles['temperature_monitoring'].to_csv(analytics_dir / 'facility_cold_chain_temperature_monitoring.csv', index=False)
cold_chain_profiles['batch_ledger'].to_csv(analytics_dir / 'facility_commodity_batch_ledger.csv', index=False)
cold_chain_profiles['expiry_risk_summary'].to_csv(analytics_dir / 'cold_chain_expiry_risk_summary.csv', index=False)


## Equipment asset, maintenance and uptime intelligence

This synthetic scenario creates an equipment asset register, monthly asset performance records, preventive/corrective maintenance work orders and facility-month summaries. Service availability excludes unplanned breakdown downtime; operational uptime also subtracts planned maintenance downtime. Preventive maintenance intervals, asset lives, failure probabilities, criticality weights and risk bands are illustrative modelling assumptions, not engineering standards or verified maintenance policies. See `docs/equipment_maintenance_methodology.md`.


In [ ]:
equipment_profiles = build_equipment_maintenance_profiles(
    facilities, periods=PERIODS, start_month=START_MONTH, seed=SEED
)
display(equipment_profiles['asset_register'].head())
display(equipment_profiles['facility_monthly_summary'].tail(20))
display(equipment_profiles['latest_asset_status'].head(20))
display(equipment_profiles['maintenance_work_orders'].head(20))


In [ ]:
analytics_dir = OUTPUT_DIR / 'analytics'
analytics_dir.mkdir(parents=True, exist_ok=True)
equipment_profiles['equipment_catalog'].to_csv(analytics_dir / 'equipment_catalog.csv', index=False)
equipment_profiles['asset_register'].to_csv(analytics_dir / 'facility_equipment_asset_register.csv', index=False)
equipment_profiles['asset_monthly_performance'].to_csv(analytics_dir / 'equipment_asset_monthly_performance.csv', index=False)
equipment_profiles['facility_monthly_summary'].to_csv(analytics_dir / 'facility_equipment_monthly_summary.csv', index=False)
equipment_profiles['maintenance_work_orders'].to_csv(analytics_dir / 'equipment_maintenance_work_orders.csv', index=False)
equipment_profiles['latest_asset_status'].to_csv(analytics_dir / 'equipment_latest_asset_risk_status.csv', index=False)
